# Northline Outfitters PDP Experiment

Evaluate whether the redesigned product detail page increases revenue per customer by at least $1.50 over the two-week test window.

Data source: `northline_pdp_test.csv`. All subsequent coding work will be completed in this notebook.

In [1]:
import random

import numpy as np
import pandas as pd

SEED = 403
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = "northline_pdp_test.csv"
df = pd.read_csv(DATA_PATH)
df.shape

(200385, 12)

## A3. Verify the randomization unit

Assignment is at the user level if every repeated `user_id` remains in one arm. The checks below also rule out region-level assignment by showing that both arms occur within every region.

In [2]:
user_check = (
    df.groupby("user_id")
      .agg(n_sessions=("session_id", "nunique"), n_arms=("arm", "nunique"))
)

randomization_check = pd.Series({
    "sessions": len(df),
    "unique_users": df["user_id"].nunique(),
    "users_with_multiple_sessions": int((user_check["n_sessions"] > 1).sum()),
    "users_observed_in_both_arms": int((user_check["n_arms"] > 1).sum()),
    "regions_containing_both_arms": int((df.groupby("region")["arm"].nunique() == 2).sum()),
    "number_of_regions": df["region"].nunique(),
})

display(randomization_check.to_frame("value"))
display(pd.crosstab(df["region"], df["arm"]))

,value
sessions,200385
unique_users,24000
users_with_multiple_sessions,22166
users_observed_in_both_arms,0
regions_containing_both_arms,4
number_of_regions,4


arm,control,treatment
region,,
Midwest,18728,19944
Northeast,22609,21606
South,27706,28781
West,30615,30396


## A4. Power and minimum detectable effect

Per the project data definition, this calculation treats `revenue_14d` as the pre-treatment 14-day revenue measure. It uses one record per randomized user and does not use session revenue or conversion. The calculation is a two-sided, two-sample normal approximation with $\alpha=0.05$, power $0.80$, a common standard deviation, and the realized treatment allocation ratio.

In [3]:
import math
from scipy.stats import norm

alpha = 0.05
power = 0.80
target_effect = 1.50

# Baseline variables and assignment are constant within user; retain one row per unit.
baseline = df[["user_id", "arm", "revenue_14d"]].drop_duplicates("user_id").copy()
assert len(baseline) == df["user_id"].nunique()
assert df.groupby("user_id")[["arm", "revenue_14d"]].nunique().max().max() == 1

arm_n = baseline["arm"].value_counts()
n_t = int(arm_n["treatment"])
n_c = int(arm_n["control"])
n_total = n_t + n_c
p_t = n_t / n_total
sigma = baseline["revenue_14d"].std(ddof=1)
z_alpha = norm.ppf(1 - alpha / 2)
z_power = norm.ppf(power)

mde = (z_alpha + z_power) * sigma * math.sqrt(1 / n_t + 1 / n_c)
required_total = math.ceil(
    ((z_alpha + z_power) * sigma / target_effect) ** 2
    / (p_t * (1 - p_t))
)
required_treatment = math.ceil(required_total * p_t)
required_control = required_total - required_treatment

power_results = pd.Series({
    "treatment_users": n_t,
    "control_users": n_c,
    "pre_revenue_14d_sd_usd": sigma,
    "mde_usd": mde,
    "target_effect_usd": target_effect,
    "required_total_users": required_total,
    "required_treatment_users_approx": required_treatment,
    "required_control_users_approx": required_control,
    "capable_of_detecting_target": mde <= target_effect,
})
power_results.to_frame("value")

,value
treatment_users,12000
control_users,12000
pre_revenue_14d_sd_usd,161.546163
mde_usd,5.842852
target_effect_usd,1.5
required_total_users,364149
required_treatment_users_approx,182075
required_control_users_approx,182074
capable_of_detecting_target,False


## B1. Validity checks

All checks use the 24,000 randomized users. `revenue_14d` is treated as `pre_revenue_14d`. Because the CSV contains `pre_sessions_56d` rather than `pre_sessions_14d`, session counts are divided by four to create a 14-day equivalent; this rescaling does not change its standardized difference. Signed standardized differences are treatment minus control.

In [4]:
from scipy.stats import chisquare

# Confirm that user-level fields do not change across a user's sessions.
user_level_fields = ["arm", "revenue_14d", "pre_sessions_56d", "tenure_days", "device", "region"]
within_user_variation = (
    df.groupby("user_id")[user_level_fields]
      .nunique(dropna=False)
      .gt(1)
      .sum()
      .rename("users_with_multiple_values")
)
display(within_user_variation.to_frame())

user_df = df[["user_id"] + user_level_fields].drop_duplicates("user_id").copy()
user_df["pre_revenue_14d"] = user_df["revenue_14d"]
user_df["pre_sessions_14d"] = user_df["pre_sessions_56d"] / 4

# Sample-ratio-mismatch test against the intended 50/50 assignment.
arm_counts = user_df["arm"].value_counts().reindex(["control", "treatment"])
srm_stat, srm_pvalue = chisquare(arm_counts.to_numpy(), f_exp=np.repeat(len(user_df) / 2, 2))
srm_results = pd.Series({
    "control_users": int(arm_counts["control"]),
    "treatment_users": int(arm_counts["treatment"]),
    "chi_square": srm_stat,
    "p_value": srm_pvalue,
})
display(srm_results.to_frame("value"))

,users_with_multiple_values
arm,0
revenue_14d,0
pre_sessions_56d,0
tenure_days,0
device,0
region,0


,value
control_users,12000.0
treatment_users,12000.0
chi_square,0.0
p_value,1.0


In [5]:
def continuous_smd(data, variable):
    treatment = data.loc[data["arm"] == "treatment", variable]
    control = data.loc[data["arm"] == "control", variable]
    pooled_sd = np.sqrt((treatment.var(ddof=1) + control.var(ddof=1)) / 2)
    return treatment.mean(), control.mean(), (treatment.mean() - control.mean()) / pooled_sd

def binary_smd(p_treatment, p_control):
    pooled_p = (p_treatment + p_control) / 2
    return (p_treatment - p_control) / np.sqrt(pooled_p * (1 - pooled_p))

balance_rows = []
for variable, label in {
    "pre_revenue_14d": "Pre-revenue 14d (USD)",
    "pre_sessions_14d": "Pre-sessions 14d",
    "tenure_days": "Tenure (days)",
}.items():
    mean_t, mean_c, smd = continuous_smd(user_df, variable)
    balance_rows.append({"variable": label, "treatment": mean_t, "control": mean_c, "smd": smd})

for variable, label in [("device", "Device"), ("region", "Region")]:
    for level in sorted(user_df[variable].dropna().unique()):
        p_treatment = (user_df.loc[user_df["arm"] == "treatment", variable] == level).mean()
        p_control = (user_df.loc[user_df["arm"] == "control", variable] == level).mean()
        balance_rows.append({
            "variable": f"{label}: {level}",
            "treatment": p_treatment,
            "control": p_control,
            "smd": binary_smd(p_treatment, p_control),
        })

balance_table = pd.DataFrame(balance_rows)
balance_table["abs_smd"] = balance_table["smd"].abs()
balance_table

,variable,treatment,control,smd,abs_smd
0,Pre-revenue 14d (USD),94.490282,90.686033,0.023550,0.023550
1,Pre-sessions 14d,8.353813,8.279271,0.011238,0.011238
2,Tenure (days),517.477333,518.245167,-0.002117,0.002117
3,Device: desktop,0.363583,0.377583,-0.028988,0.028988
4,Device: mobile,0.636417,0.622417,0.028988,0.028988
5,Region: Midwest,0.196250,0.189583,0.016895,0.016895
6,Region: Northeast,0.216833,0.220583,-0.009072,0.009072
7,Region: South,0.282667,0.284750,-0.004621,0.004621
8,Region: West,0.304250,0.305083,-0.001811,0.001811


In [6]:
# Missingness and identifier duplication checks.
missing_by_column = df.isna().sum().rename("missing_rows").to_frame()
identifier_checks = pd.Series({
    "rows": len(df),
    "unique_user_ids": df["user_id"].nunique(dropna=False),
    "rows_with_repeated_user_id": int(df["user_id"].duplicated(keep=False).sum()),
    "unique_session_ids": df["session_id"].nunique(dropna=False),
    "duplicated_session_id_rows_beyond_first": int(df["session_id"].duplicated().sum()),
    "duplicated_user_session_pairs_beyond_first": int(df.duplicated(["user_id", "session_id"]).sum()),
    "exact_duplicate_rows_beyond_first": int(df.duplicated().sum()),
    "users_in_multiple_arms": int((df.groupby("user_id")["arm"].nunique() > 1).sum()),
})
display(missing_by_column)
display(identifier_checks.to_frame("value"))

,missing_rows
user_id,0
session_id,0
arm,0
session_date,0
revenue,0
converted,0
device,0
region,0
tenure_days,0
revenue_14d,0


,value
rows,200385
unique_user_ids,24000
rows_with_repeated_user_id,198551
unique_session_ids,200385
duplicated_session_id_rows_beyond_first,0
duplicated_user_session_pairs_beyond_first,0
exact_duplicate_rows_beyond_first,0
users_in_multiple_arms,0


## B2. Naive session-level estimate

This deliberately naive analysis treats every delivered row as an independent observation. It regresses session revenue on a treatment indicator and uses an HC1 heteroskedasticity-robust standard error. Because users contribute multiple sessions, this is a session-weighted contrast and the standard error does not account for within-user dependence; B3 will use it as the required benchmark.

In [7]:
import statsmodels.api as sm

naive_df = df[["revenue", "arm"]].copy()
naive_df["treatment"] = (naive_df["arm"] == "treatment").astype(int)
naive_model = sm.OLS(
    naive_df["revenue"],
    sm.add_constant(naive_df["treatment"]),
).fit(cov_type="HC1")

naive_ci = naive_model.conf_int(alpha=0.05).loc["treatment"]
naive_results = pd.Series({
    "control_sessions": int((naive_df["treatment"] == 0).sum()),
    "treatment_sessions": int((naive_df["treatment"] == 1).sum()),
    "control_mean_revenue": naive_df.loc[naive_df["treatment"] == 0, "revenue"].mean(),
    "treatment_mean_revenue": naive_df.loc[naive_df["treatment"] == 1, "revenue"].mean(),
    "difference_in_means": naive_model.params["treatment"],
    "hc1_standard_error": naive_model.bse["treatment"],
    "ci_95_lower": naive_ci.iloc[0],
    "ci_95_upper": naive_ci.iloc[1],
})
naive_results.to_frame("value")

,value
control_sessions,99658.000000
treatment_sessions,100727.000000
control_mean_revenue,10.919669
treatment_mean_revenue,11.256995
difference_in_means,0.337326
hc1_standard_error,0.139840
ci_95_lower,0.063245
ci_95_upper,0.611407


## B3. Diagnose and correct the failure

The B2 model has two related failures. First, repeated sessions from the same randomized user are dependent, so HC1 treats correlated rows as independent and overstates the effective sample size. Second, a session-level mean weights a user in proportion to their number of sessions, whereas the estimand gives every randomized user one vote. We diagnose the first problem by comparing HC1 with a user-clustered standard error for the identical session-level regression. We correct the estimand by summing session revenue within user and estimating the treatment-control difference on the 24,000 user totals.

In [8]:
# Evidence of repeated observations and unequal session weights.
user_outcomes = (
    df.groupby(["user_id", "arm"], as_index=False)
      .agg(total_revenue_14d=("revenue", "sum"), sessions=("session_id", "nunique"))
)

cluster_evidence = pd.Series({
    "users": len(user_outcomes),
    "users_with_multiple_sessions": int((user_outcomes["sessions"] > 1).sum()),
    "share_with_multiple_sessions": (user_outcomes["sessions"] > 1).mean(),
    "mean_sessions_per_user": user_outcomes["sessions"].mean(),
    "median_sessions_per_user": user_outcomes["sessions"].median(),
    "max_sessions_per_user": user_outcomes["sessions"].max(),
    "correlation_sessions_total_revenue": user_outcomes[["sessions", "total_revenue_14d"]].corr().iloc[0, 1],
})
display(cluster_evidence.to_frame("value"))
display(user_outcomes.groupby("arm")[["sessions", "total_revenue_14d"]].mean())

,value
users,24000.000000
users_with_multiple_sessions,22166.000000
share_with_multiple_sessions,0.923583
mean_sessions_per_user,8.349375
median_sessions_per_user,7.000000
max_sessions_per_user,102.000000
correlation_sessions_total_revenue,0.507033


,sessions,total_revenue_14d
arm,,
control,8.304833,90.686032
treatment,8.393917,94.490282


In [9]:
# Hold the B2 coefficient fixed and correct only its covariance for user clustering.
naive_ols = sm.OLS(
    naive_df["revenue"],
    sm.add_constant(naive_df["treatment"]),
).fit()
clustered_model = naive_ols.get_robustcov_results(
    cov_type="cluster",
    groups=df["user_id"],
    use_correction=True,
)
naive_se = float(naive_model.bse["treatment"])
clustered_se = float(clustered_model.bse[1])
se_comparison = pd.Series({
    "same_session_level_estimate": naive_model.params["treatment"],
    "naive_hc1_se": naive_se,
    "user_clustered_se": clustered_se,
    "clustered_to_naive_ratio": clustered_se / naive_se,
    "corrected_se_percent_larger": 100 * (clustered_se / naive_se - 1),
    "naive_se_percent_below_corrected": 100 * (1 - naive_se / clustered_se),
})
se_comparison.to_frame("value")

,value
same_session_level_estimate,0.337326
naive_hc1_se,0.139840
user_clustered_se,0.215366
clustered_to_naive_ratio,1.540093
corrected_se_percent_larger,54.009316
naive_se_percent_below_corrected,35.068863


In [10]:
# Correct target analysis: one two-week revenue total per randomized user.
user_outcomes["treatment"] = (user_outcomes["arm"] == "treatment").astype(int)
corrected_model = sm.OLS(
    user_outcomes["total_revenue_14d"],
    sm.add_constant(user_outcomes["treatment"]),
).fit(cov_type="HC1")
corrected_ci = corrected_model.conf_int(alpha=0.05).loc["treatment"]
corrected_results = pd.Series({
    "control_users": int((user_outcomes["treatment"] == 0).sum()),
    "treatment_users": int((user_outcomes["treatment"] == 1).sum()),
    "control_mean_total_revenue": user_outcomes.loc[user_outcomes["treatment"] == 0, "total_revenue_14d"].mean(),
    "treatment_mean_total_revenue": user_outcomes.loc[user_outcomes["treatment"] == 1, "total_revenue_14d"].mean(),
    "corrected_customer_level_effect": corrected_model.params["treatment"],
    "corrected_hc1_standard_error": corrected_model.bse["treatment"],
    "corrected_ci_95_lower": corrected_ci.iloc[0],
    "corrected_ci_95_upper": corrected_ci.iloc[1],
})
corrected_results.to_frame("value")

,value
control_users,12000.000000
treatment_users,12000.000000
control_mean_total_revenue,90.686033
treatment_mean_total_revenue,94.490282
corrected_customer_level_effect,3.804249
corrected_hc1_standard_error,2.085451
corrected_ci_95_lower,-0.283159
corrected_ci_95_upper,7.891658


## B4. CUPED variance reduction

CUPED must use a variable measured before treatment. The first check below tests the timing interpretation against the logged data. It verifies whether `revenue_14d` is distinct from, or is exactly the same as, the two-week outcome constructed by summing session revenue. Because outcome leakage would make CUPED invalid, the adjustment uses `pre_revenue_56d`, the revenue field explicitly marked as pre-period in the CSV. Scaling that field to 14 days would not change an OLS/CUPED adjustment.

In [11]:
preperiod_user = (
    df[["user_id", "revenue_14d", "pre_revenue_56d"]]
      .drop_duplicates("user_id")
)
cuped_df = user_outcomes.merge(preperiod_user, on="user_id", validate="one_to_one")
outcome_difference = cuped_df["total_revenue_14d"] - cuped_df["revenue_14d"]
timing_check = pd.Series({
    "users": len(cuped_df),
    "revenue_14d_equals_summed_session_revenue_for_all_users": bool(np.allclose(outcome_difference, 0)),
    "maximum_absolute_difference": outcome_difference.abs().max(),
    "correlation_outcome_with_revenue_14d": cuped_df[["total_revenue_14d", "revenue_14d"]].corr().iloc[0, 1],
    "correlation_outcome_with_pre_revenue_56d": cuped_df[["total_revenue_14d", "pre_revenue_56d"]].corr().iloc[0, 1],
})
timing_check.to_frame("value")

,value
users,24000
revenue_14d_equals_summed_session_revenue_for_all_users,True
maximum_absolute_difference,0.0
correlation_outcome_with_revenue_14d,1.0
correlation_outcome_with_pre_revenue_56d,0.687791


In [12]:
# Equivalent CUPED/ANCOVA specification at the randomized-user level.
cuped_df["pre_revenue_56d_centered"] = (
    cuped_df["pre_revenue_56d"] - cuped_df["pre_revenue_56d"].mean()
)
cuped_model = sm.OLS(
    cuped_df["total_revenue_14d"],
    sm.add_constant(cuped_df[["treatment", "pre_revenue_56d_centered"]]),
).fit(cov_type="HC1")
cuped_ci = cuped_model.conf_int(alpha=0.05).loc["treatment"]

unadjusted_variance = corrected_model.bse["treatment"] ** 2
cuped_variance = cuped_model.bse["treatment"] ** 2
variance_reduction = 1 - cuped_variance / unadjusted_variance
cuped_mde = mde * np.sqrt(cuped_variance / unadjusted_variance)

cuped_results = pd.Series({
    "pre_revenue_56d_control_mean": cuped_df.loc[cuped_df["treatment"] == 0, "pre_revenue_56d"].mean(),
    "pre_revenue_56d_treatment_mean": cuped_df.loc[cuped_df["treatment"] == 1, "pre_revenue_56d"].mean(),
    "pre_revenue_coefficient": cuped_model.params["pre_revenue_56d_centered"],
    "unadjusted_effect": corrected_model.params["treatment"],
    "unadjusted_hc1_se": corrected_model.bse["treatment"],
    "cuped_effect": cuped_model.params["treatment"],
    "cuped_hc1_se": cuped_model.bse["treatment"],
    "cuped_ci_95_lower": cuped_ci.iloc[0],
    "cuped_ci_95_upper": cuped_ci.iloc[1],
    "variance_reduction_percent": 100 * variance_reduction,
    "a4_unadjusted_mde": mde,
    "cuped_mde_if_preplanned": cuped_mde,
    "mde_reduction_percent": 100 * (1 - cuped_mde / mde),
})
cuped_results.to_frame("value")

,value
pre_revenue_56d_control_mean,367.289108
pre_revenue_56d_treatment_mean,360.438442
pre_revenue_coefficient,0.184431
unadjusted_effect,3.804249
unadjusted_hc1_se,2.085451
cuped_effect,5.067726
cuped_hc1_se,1.509855
cuped_ci_95_lower,2.108464
cuped_ci_95_upper,8.026988
variance_reduction_percent,47.583180


## B5. Randomization inference

Under the sharp null, every user's observed two-week revenue would be unchanged under either assignment. The randomization distribution is generated by reallocating exactly 12,000 of the 24,000 users to treatment, preserving the experimental 50/50 design. The statistic is the absolute user-level difference in mean revenue, and the seed defined at the top makes the Monte Carlo result reproducible.

In [13]:
n_permutations = 10_000
rng = np.random.default_rng(SEED)
y_ri = user_outcomes["total_revenue_14d"].to_numpy()
z_ri = user_outcomes["treatment"].to_numpy()
n_users_ri = len(y_ri)
n_treatment_ri = int(z_ri.sum())
observed_ri = y_ri[z_ri == 1].mean() - y_ri[z_ri == 0].mean()
total_y_ri = y_ri.sum()

permuted_effects = np.empty(n_permutations)
for b in range(n_permutations):
    treated_indices = rng.choice(n_users_ri, size=n_treatment_ri, replace=False)
    treated_sum = y_ri[treated_indices].sum()
    permuted_effects[b] = (
        treated_sum / n_treatment_ri
        - (total_y_ri - treated_sum) / (n_users_ri - n_treatment_ri)
    )

extreme_count = int((np.abs(permuted_effects) >= abs(observed_ri)).sum())
permutation_pvalue = (extreme_count + 1) / (n_permutations + 1)
analytical_pvalue = float(corrected_model.pvalues["treatment"])
monte_carlo_se = np.sqrt(permutation_pvalue * (1 - permutation_pvalue) / (n_permutations + 1))

ri_results = pd.Series({
    "observed_user_level_effect": observed_ri,
    "permutations": n_permutations,
    "extreme_permutations": extreme_count,
    "two_sided_permutation_p_value": permutation_pvalue,
    "permutation_p_value_monte_carlo_se": monte_carlo_se,
    "two_sided_analytical_hc1_p_value": analytical_pvalue,
    "absolute_p_value_difference": abs(permutation_pvalue - analytical_pvalue),
})
ri_results.to_frame("value")

,value
observed_user_level_effect,3.804249
permutations,10000.000000
extreme_permutations,688.000000
two_sided_permutation_p_value,0.068893
permutation_p_value_monte_carlo_se,0.002533
two_sided_analytical_hc1_p_value,0.068124
absolute_p_value_difference,0.000769
